# Step 4 â€” Trio inheritance (chr22, GIAB HG002/HG003/HG004)

Track 1 (exact-identity inheritance) from the handoff: build the child's (HG002) two haplotype CDS
sequences, and check each against the parents' (HG003 father / HG004 mother) genotypes to assign
parent-of-origin.

**Method note -- no `bcftools consensus` / whole-genome FASTA needed.** Rather than materializing a
full consensus genome (which would require downloading the ~850MB same-release GRCh38 genome FASTA),
variants are overlaid directly onto the already-validated reference exon sequences from notebook 02,
using the GTF's genomic CDS-exon coordinates to map each VCF position to a coding-relative offset.
This reuses 100% of the validated notebook 02/03 work and keeps everything running in plain Windows
Python -- no WSL kernel switch needed for this step either, since the chr22-scoped GIAB VCFs (~2MB
each) are small enough to parse entirely with the stdlib `gzip` module.

**Parent-of-origin check is a per-site membership test, not haplotype enumeration.** A child haplotype
is "explainable by parent X" if, at every variant position in that CDS, the child's allele is one of
parent X's two alleles at that site. This needs no parental phasing at all -- only the child needs to be
phased (to know which alleles co-occur on one physical chromosome copy in the first place).

**Scope limits, stated up front (matches the project's "flag as QC, don't guess" rule):**
- **SNVs only.** Any CDS with an overlapping indel (in any of the 3 individuals) is flagged
  `has_indel` and excluded -- indels shift downstream coding-relative offsets and combine badly with
  the strand-complement math; out of scope for this increment.
- **Child haplotypes need phase.** 0-1 heterozygous sites per CDS need no phase info (unambiguous
  either way). 2+ heterozygous sites require all of them to share one phased block (`PS` + `|`
  genotype separator); otherwise flagged `phase_incomplete`.
- **GIAB's own `_benchmark_noinconsistent.bed` exclusion file was not downloaded/applied** -- so a
  small number of known Mendelian-inconsistent benchmark sites are expected to surface as
  `no_parental_match`, not as real de novo calls (see the worked investigation below).

In [1]:
import gzip
import os
import sqlite3
import sys
from collections import defaultdict

sys.path.insert(0, os.path.abspath("../scripts"))
import gencode_cds_extract as cds

REF = "../data/reference"
GIAB = "../data/giab"
DB_PATH = "../data/derived/chr22/hash_catalog.db"
CHROM = "chr22"

COMPLEMENT = str.maketrans("ACGT", "TGCA")
def revcomp(s):
    return s.translate(COMPLEMENT)[::-1]

## Load the trio VCFs

Each file is a single-sample, chr22-only slice (fetched via remote `bcftools view -r chr22`, not a
full-genome download -- see `data/giab/manifest.tsv`). Keyed by genomic position; `PS` (phase set) is
only meaningful for HG002, whose file is the trio+StrandSeq phased release.

In [2]:
def load_vcf(path):
    """pos -> dict(ref, alt=[...], gt=(a,b), phased, ps, difficultregion). Non-PASS and
    missing-GT rows are dropped. difficultregion is GIAB's own INFO annotation for
    segmental-duplication / low-mappability / tandem-repeat regions prone to mapping
    artifacts -- kept as metadata, not used to filter, per the handoff's "flag, don't
    guess" rule."""
    variants = {}
    with gzip.open(path, "rt") as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            f = line.rstrip("\n").split("\t")
            chrom, pos, _id, ref, alt, qual, flt, info, fmt, sample = f[:10]
            if flt not in ("PASS", "."):
                continue
            fmt_fields = fmt.split(":")
            sample_fields = sample.split(":")
            gt_raw = sample_fields[fmt_fields.index("GT")]
            ps = sample_fields[fmt_fields.index("PS")] if "PS" in fmt_fields else None
            phased = "|" in gt_raw
            sep = "|" if phased else "/"
            try:
                a, b = gt_raw.split(sep)
                gt = (int(a), int(b))
            except ValueError:
                continue
            difficultregion = None
            for kv in info.split(";"):
                if kv.startswith("difficultregion="):
                    difficultregion = kv[len("difficultregion="):]
            variants[int(pos)] = {
                "ref": ref, "alt": alt.split(","), "gt": gt, "phased": phased,
                "ps": ps if ps not in (None, ".") else None,
                "difficultregion": difficultregion,
            }
    return variants

vcf = {
    "HG002": load_vcf(os.path.join(GIAB, "HG002_chr22_phased.vcf.gz")),
    "HG003": load_vcf(os.path.join(GIAB, "HG003_chr22.vcf.gz")),
    "HG004": load_vcf(os.path.join(GIAB, "HG004_chr22.vcf.gz")),
}
for s, v in vcf.items():
    tagged = sum(1 for x in v.values() if x["difficultregion"])
    print(f"{s}: {len(v)} PASS variants ({tagged} tagged difficultregion, {tagged/len(v):.1%})")


HG002: 50284 PASS variants (13576 tagged difficultregion, 27.0%)
HG003: 50898 PASS variants (13391 tagged difficultregion, 26.3%)
HG004: 47281 PASS variants (12560 tagged difficultregion, 26.6%)


## Rebuild the validated reference catalog (notebook 02/03)

Same call as notebook 03 -- reuses the promoted, already-validated extraction (1341/1398 chr22
transcripts).

In [3]:
chrom_transcripts = cds.parse_gtf_chrom(os.path.join(REF, "gencode.v46.basic.annotation.gtf.gz"), CHROM)
tx_seqs, tx_meta = cds.load_transcripts_fasta(os.path.join(REF, "gencode.v46.pc_transcripts.fa.gz"))
prot_seqs, protein_ids = cds.load_translations_fasta(os.path.join(REF, "gencode.v46.pc_translations.fa.gz"))
catalog, flagged = cds.build_catalog(chrom_transcripts, tx_seqs, tx_meta, prot_seqs, protein_ids)
print(f"validated reference transcripts: {len(catalog)}")

validated reference transcripts: 1341


## Map VCF positions onto coding-relative offsets

For each CDS-exon genomic block, compute each variant's offset into the whole (stop-excluded) CDS
string, complementing REF/ALT for minus-strand genes (the CDS string is already in mRNA sense).
Any multi-allelic or indel site in range flags the whole transcript as `has_indel` (excluded, not
silently mishandled).

In [4]:
def variants_in_range(sample_vcf, start, end):
    return [(pos, v) for pos, v in sample_vcf.items() if start <= pos <= end]

def build_transcript_variant_map(entry):
    t = chrom_transcripts[entry["transcript_id"]]
    strand = t["strand"]
    blocks = t["cds"]

    block_offsets, running = [], 0
    for s, e in blocks:
        block_offsets.append(running)
        running += e - s + 1

    has_indel = False
    per_sample_offsets = {name: {} for name in vcf}
    for (s, e), block_offset in zip(blocks, block_offsets):
        for sample_name, sample_vcf in vcf.items():
            for pos, v in variants_in_range(sample_vcf, s, e):
                ref, alts = v["ref"], v["alt"]
                if len(alts) != 1 or len(ref) != 1 or len(alts[0]) != 1:
                    has_indel = True
                    continue
                alt = alts[0]
                if strand == "+":
                    coding_pos = block_offset + (pos - s)
                    ref_c, alt_c = ref, alt
                else:
                    coding_pos = block_offset + (e - pos)
                    ref_c, alt_c = revcomp(ref), revcomp(alt)
                per_sample_offsets[sample_name][coding_pos] = {
                    "ref": ref_c, "alt": alt_c, "gt": v["gt"], "phased": v["phased"], "ps": v["ps"],
                    "difficultregion": v["difficultregion"],
                }
    return per_sample_offsets, has_indel


### Sanity check: REF-base consistency

Before trusting the offset math, confirm the strand-corrected REF allele from each VCF actually matches
the known reference CDS base at that coding position, across all SNV-only transcripts. Any mismatch
would mean the coordinate mapping (or strand handling) is wrong.

In [5]:
ref_mismatches = 0
snv_only_count = 0
indel_count = 0

for entry in catalog:
    per_sample_offsets, has_indel = build_transcript_variant_map(entry)
    if has_indel:
        indel_count += 1
        continue
    snv_only_count += 1
    cds_seq = entry["cds_seq"]
    for offsets in per_sample_offsets.values():
        for coding_pos, v in offsets.items():
            if 0 <= coding_pos < len(cds_seq) and cds_seq[coding_pos] != v["ref"]:
                ref_mismatches += 1

print(f"transcripts with an indel in range (flagged, excluded): {indel_count}")
print(f"transcripts checked (SNV-only): {snv_only_count}")
print(f"REF-base mismatches: {ref_mismatches}")
assert ref_mismatches == 0, "coordinate/strand mapping is wrong -- stop and investigate"

transcripts with an indel in range (flagged, excluded): 30
transcripts checked (SNV-only): 1311
REF-base mismatches: 0


## Build child haplotypes; check parent membership

`build_child_haplotypes`: homozygous sites apply to both haplotypes unconditionally. 0-1 heterozygous
sites need no phase info. 2+ heterozygous sites must share one phased block, else `phase_incomplete`.

`explainable_by`: per-site membership check against a parent's genotype -- no parental phasing needed.

In [6]:
def build_child_haplotypes(cds_seq, child_offsets):
    het_sites = {p: v for p, v in child_offsets.items() if v["gt"][0] != v["gt"][1]}
    if len(het_sites) >= 2:
        ps_values = {v["ps"] for v in het_sites.values()}
        all_phased = all(v["phased"] for v in het_sites.values())
        if not all_phased or len(ps_values) != 1 or None in ps_values:
            return None, None, "phase_incomplete"

    hap0, hap1 = list(cds_seq), list(cds_seq)
    for p, v in child_offsets.items():
        alleles = (v["ref"], v["alt"])
        hap0[p] = alleles[v["gt"][0]]
        hap1[p] = alleles[v["gt"][1]]
    return "".join(hap0), "".join(hap1), "ok"

def parent_allele_set(parent_offsets, pos, ref_base):
    if pos not in parent_offsets:
        return {ref_base}
    v = parent_offsets[pos]
    alleles = (v["ref"], v["alt"])
    return {alleles[v["gt"][0]], alleles[v["gt"][1]]}

def explainable_by(parent_offsets, hap_seq, cds_seq, variant_positions):
    return all(hap_seq[p] in parent_allele_set(parent_offsets, p, cds_seq[p]) for p in variant_positions)

In [7]:
categories = defaultdict(int)
results = []  # (transcript_id, gene_id, hap_name, category, md5, sq, difficult_region)

for entry in catalog:
    per_sample_offsets, has_indel = build_transcript_variant_map(entry)
    if has_indel:
        categories["has_indel"] += 1
        continue

    cds_seq = entry["cds_seq"]
    hap0, hap1, status = build_child_haplotypes(cds_seq, per_sample_offsets["HG002"])
    if status != "ok":
        categories["phase_incomplete"] += 1
        continue

    if not per_sample_offsets["HG002"]:
        categories["no_variants_in_child"] += 1
        continue

    variant_positions = sorted(
        set(per_sample_offsets["HG002"]) | set(per_sample_offsets["HG003"]) | set(per_sample_offsets["HG004"])
    )
    child_difficult = any(v["difficultregion"] for v in per_sample_offsets["HG002"].values())

    for hap_name, hap_seq in [("hap0", hap0), ("hap1", hap1)]:
        by_father = explainable_by(per_sample_offsets["HG003"], hap_seq, cds_seq, variant_positions)
        by_mother = explainable_by(per_sample_offsets["HG004"], hap_seq, cds_seq, variant_positions)
        if by_father and not by_mother:
            cat = "paternal_origin"
        elif by_mother and not by_father:
            cat = "maternal_origin"
        elif by_father and by_mother:
            cat = "uninformative_shared"
        else:
            cat = "no_parental_match"
        categories[cat] += 1
        results.append((entry["transcript_id"], entry["gene_id"], hap_name, cat,
                         cds.md5_digest(hap_seq), cds.ga4gh_sq_digest(hap_seq), child_difficult))

print("=== classification summary (per haplotype, where applicable) ===")
for cat, n in sorted(categories.items(), key=lambda kv: -kv[1]):
    print(f"  {n:5d}  {cat}")

print()
print("=== difficult-region overlap, by category ===")
by_cat_difficult = defaultdict(lambda: [0, 0])
for _, _, _, cat, _, _, difficult in results:
    by_cat_difficult[cat][0] += 1
    by_cat_difficult[cat][1] += int(difficult)
for cat, (total, difficult) in sorted(by_cat_difficult.items(), key=lambda kv: -kv[1][0]):
    print(f"  {cat:20s} {difficult:4d}/{total:4d} ({difficult/total:.1%}) involve a difficultregion-tagged site")


=== classification summary (per haplotype, where applicable) ===
    681  no_variants_in_child
    675  uninformative_shared
    230  maternal_origin
    201  paternal_origin
     75  phase_incomplete
     30  has_indel
      4  no_parental_match

=== difficult-region overlap, by category ===
  uninformative_shared   59/ 675 (8.7%) involve a difficultregion-tagged site
  maternal_origin        18/ 230 (7.8%) involve a difficultregion-tagged site
  paternal_origin        15/ 201 (7.5%) involve a difficultregion-tagged site
  no_parental_match       4/   4 (100.0%) involve a difficultregion-tagged site


## Worked examples: confident parent-of-origin calls

Two transcripts of the same gene (`ENSG00000177663`) independently agree on which haplotype is
paternal vs maternal -- a nice internal consistency check across isoforms.

In [8]:
shown = defaultdict(int)
for tid, gid, hap, cat, md5, sq, difficult in results:
    if cat in ("paternal_origin", "maternal_origin") and shown[cat] < 3:
        flag = " [difficultregion]" if difficult else ""
        print(f"  {cat:16s} {tid} ({gid}) {hap}  MD5={md5}{flag}")
        shown[cat] += 1


  paternal_origin  ENST00000319363.11 (ENSG00000177663.15) hap0  MD5=186a8719099628f01f1727701e6841e1
  maternal_origin  ENST00000319363.11 (ENSG00000177663.15) hap1  MD5=1584973f60cb3c10db2f8dc642b865d9
  paternal_origin  ENST00000612619.2 (ENSG00000177663.15) hap0  MD5=694e824230f7ac4a2cbd3d44ec44a4d3
  maternal_origin  ENST00000612619.2 (ENSG00000177663.15) hap1  MD5=9e7ad7ccecd0c07e7dd2874212db48b8
  maternal_origin  ENST00000155674.9 (ENSG00000069998.12) hap1  MD5=8f0e6b8cdc8b70df86d14b19c262a7f1
  paternal_origin  ENST00000252137.11 (ENSG00000100056.12) hap0  MD5=d839a0ea397aa5d12695ee27054cd50f


## Investigating `no_parental_match`

Per the handoff: "matches neither parent" is almost always artifact, not real de novo. Confirming that
here rather than asserting it.

In [9]:
no_match_details = []
for entry in catalog:
    per_sample_offsets, has_indel = build_transcript_variant_map(entry)
    if has_indel:
        continue
    cds_seq = entry["cds_seq"]
    hap0, hap1, status = build_child_haplotypes(cds_seq, per_sample_offsets["HG002"])
    if status != "ok" or not per_sample_offsets["HG002"]:
        continue
    variant_positions = sorted(
        set(per_sample_offsets["HG002"]) | set(per_sample_offsets["HG003"]) | set(per_sample_offsets["HG004"])
    )
    for hap_name, hap_seq in [("hap0", hap0), ("hap1", hap1)]:
        by_father = explainable_by(per_sample_offsets["HG003"], hap_seq, cds_seq, variant_positions)
        by_mother = explainable_by(per_sample_offsets["HG004"], hap_seq, cds_seq, variant_positions)
        if not by_father and not by_mother:
            mismatch_pos = [p for p in variant_positions
                            if hap_seq[p] not in parent_allele_set(per_sample_offsets["HG003"], p, cds_seq[p])
                            and hap_seq[p] not in parent_allele_set(per_sample_offsets["HG004"], p, cds_seq[p])]
            for p in mismatch_pos:
                child_v = per_sample_offsets["HG002"].get(p)
                in_father_vcf = p in per_sample_offsets["HG003"]
                in_mother_vcf = p in per_sample_offsets["HG004"]
                no_match_details.append((entry["transcript_id"], entry["gene_id"], hap_name, p,
                                          child_v["difficultregion"], in_father_vcf, in_mother_vcf))

for tid, gid, hap, pos, difficultregion, in_father, in_mother in no_match_details:
    print(f"{tid} ({gid}) {hap} pos {pos}: father_has_call={in_father} mother_has_call={in_mother} "
          f"difficultregion={difficultregion}")

distinct_genes = {gid for _, gid, _, _, _, _, _ in no_match_details}
print(f"\n{len(no_match_details)} flagged haplotypes -> {len(distinct_genes)} distinct gene(s): {distinct_genes}")
print("Neither parent has ANY VCF record at this position (both implied homozygous-reference by absence),")
print("while the child is heterozygous 0/1 -- and the site carries GIAB's own 'difficultregion' tag")
print("(segmental duplication + low mappability). That combination is the signature of a mapping")
print("artifact (reads from a paralogous locus mismapping here), not a true de novo mutation --")
print("consistent with the handoff's own caution that 'matches neither parent' is almost always artifact.")


ENST00000610940.4 (ENSG00000100033.17) hap1 pos 1740: father_has_call=False mother_has_call=False difficultregion=hg38.segdups_sorted_merged,lowmappabilityall
ENST00000357068.11 (ENSG00000100033.17) hap1 pos 1740: father_has_call=False mother_has_call=False difficultregion=hg38.segdups_sorted_merged,lowmappabilityall
ENST00000420436.5 (ENSG00000100033.17) hap1 pos 1416: father_has_call=False mother_has_call=False difficultregion=hg38.segdups_sorted_merged,lowmappabilityall
ENST00000334029.6 (ENSG00000100033.17) hap1 pos 1416: father_has_call=False mother_has_call=False difficultregion=hg38.segdups_sorted_merged,lowmappabilityall

4 flagged haplotypes -> 1 distinct gene(s): {'ENSG00000100033.17'}
Neither parent has ANY VCF record at this position (both implied homozygous-reference by absence),
while the child is heterozygous 0/1 -- and the site carries GIAB's own 'difficultregion' tag
(segmental duplication + low mappability). That combination is the signature of a mapping
artifact (rea

## Persist to the SQLite catalog

Reuses the existing `sequences` schema (no schema change needed) -- child haplotype rows are just more
`seq_type='CDS'` rows, distinguished by accession (`{transcript_id}.HG002.{hap}`) and `source='GIAB'`,
with the inheritance classification recorded in `evidence`.

In [10]:
conn = sqlite3.connect(DB_PATH)
conn.execute("DELETE FROM sequences WHERE source = 'GIAB'")  # idempotent re-run

length_by_key = {}
for entry in catalog:
    per_sample_offsets, has_indel = build_transcript_variant_map(entry)
    if has_indel:
        continue
    length_by_key[entry["transcript_id"]] = len(entry["cds_seq"])

rows = [
    (md5, sq, "CDS", f"{tid}.HG002.{hap}", gid, "GIAB", "NISTv4.2.1",
     f"classification={cat};difficult_region={difficult}", length_by_key.get(tid), None)
    for tid, gid, hap, cat, md5, sq, difficult in results
]

conn.executemany(
    "INSERT INTO sequences (hash_md5, hash_sq, seq_type, accession, gene_id, source, release, evidence, length, low_complexity_frac) "
    "VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
    rows,
)
conn.commit()
print(f"inserted {len(rows)} HG002 haplotype rows into {DB_PATH}")

counts = conn.execute(
    "SELECT evidence, COUNT(*) FROM sequences WHERE source='GIAB' GROUP BY evidence ORDER BY 2 DESC LIMIT 10"
).fetchall()
for evidence, n in counts:
    print(f"  {n:5d}  {evidence}")


inserted 1110 HG002 haplotype rows into ../data/derived/chr22/hash_catalog.db
    616  classification=uninformative_shared;difficult_region=False
    212  classification=maternal_origin;difficult_region=False
    186  classification=paternal_origin;difficult_region=False
     59  classification=uninformative_shared;difficult_region=True
     18  classification=maternal_origin;difficult_region=True
     15  classification=paternal_origin;difficult_region=True
      4  classification=no_parental_match;difficult_region=True


## Summary

Out of 1341 validated chr22 protein-coding transcripts:

- **431 confident parent-of-origin calls** (201 paternal + 230 maternal haplotypes)
- **675 uninformative** (variant shared by both parents -- can't distinguish, per the handoff's own
  Mendelian-uniqueness caveat)
- **681 haplotype-pairs trivially identical** (no variants in the child for that CDS at all)
- **75 flagged `phase_incomplete`** (child heterozygous sites not resolvably phased together)
- **30 flagged `has_indel`** (out of scope for this increment)
- **4 flagged `no_parental_match`**, all four from one gene/locus: neither parent has *any* VCF record
  at that position (implied homozygous-reference), the child is heterozygous, and the site carries
  GIAB's own `difficultregion=hg38.segdups_sorted_merged,lowmappabilityall` tag -- the signature of a
  mapping artifact (reads from a paralogous locus), not a real de novo mutation. (Earlier notes described
  this as "both parents C/C" -- that was imprecise; corrected here after checking the raw VCF records.)

`difficultregion` (GIAB's own INFO annotation, no extra download needed) is now tracked per haplotype as
metadata -- see the per-category breakdown above for how many of the 431 confident calls also touch a
difficult region. Consistent with the handoff's "store as separate metadata, don't discard" rule: these
calls are kept, just annotated, so their confidence can be judged rather than silently trusted or dropped.

## Next steps

1. Extend variant handling to indels (the 30 `has_indel` transcripts) -- needs coordinate-shift-aware
   handling per variant, ordered by position within each exon.
2. Low-complexity flagging (`dustmasker`/`segmasker`) to populate `low_complexity_frac` (still NULL).
3. *(Deferred, Track 2)* Mash/sourmash population-distance demo.

(The originally planned "apply GIAB's `_benchmark_noinconsistent.bed`" step turned out not to be the
right tool here -- checking GIAB's own README showed that BED is just the primary high-confidence
region file already implicitly used via the benchmark VCF's PASS filter, not a separate trio-consistency
filter. The `difficultregion` INFO tag, already present in the VCF we have, is the more direct signal.)
